# PC.23 - Reconhecimento de Fala em Inglês

Transcrição de fala em **língua inglesa** (`en-US`) a partir de um **arquivo de áudio** (.wav, .aiff, .flac) usando a biblioteca `SpeechRecognition` com o serviço Google.

> **Pré-requisito:** conexão com a internet (o método `recognize_google` é online).

## 1. Instalação das dependências

In [ ]:
%pip install SpeechRecognition>=3.10.0

## 2. Importações

In [ ]:
from pathlib import Path
import speech_recognition as sr

## 3. Função de transcrição

In [ ]:
def transcribe_from_file(recognizer: sr.Recognizer, audio_path: Path) -> str:
    """Transcribe speech from an audio file (.wav, .aiff, .flac)."""
    with sr.AudioFile(str(audio_path)) as source:
        audio = recognizer.record(source)
    return recognizer.recognize_google(audio, language="en-US")

## 4. Executar a transcrição

Altere o valor de `AUDIO_PATH` para o caminho do seu arquivo de áudio.

In [ ]:
AUDIO_PATH = Path("sample.wav")  # <- altere para o caminho do seu arquivo

recognizer = sr.Recognizer()

try:
    if not AUDIO_PATH.exists():
        raise FileNotFoundError(f"Arquivo de áudio não encontrado: {AUDIO_PATH}")

    text = transcribe_from_file(recognizer, AUDIO_PATH)

    print("--- TRANSCRIÇÃO (en-US) ---")
    print(text)

except sr.UnknownValueError:
    print("Não foi possível entender o áudio (fala não reconhecida).")
except sr.RequestError as exc:
    print(f"Serviço de reconhecimento indisponível ou falha na requisição: {exc}")
except Exception as exc:
    print(f"Erro: {exc}")

## 5. Casos problemáticos para PLN

Exemplos de situações em que a transcrição automática pode gerar saída problemática para um pipeline de PLN.

---

### Caso A — Homófonos e pontuação ausente

| | Texto |
|---|---|
| **Fala esperada** | `Let's eat, Grandma.` |
| **Possível transcrição ASR** | `lets eat grandma` |

**Problemas para PLN:**
- A perda de pontuação altera o sentido semântico.
- Tokenização e análise sintática podem interpretar uma sentença ofensiva/errada.
- Tarefas de classificação (ex.: toxicidade) podem receber falso positivo.

---

### Caso B — Entidades nomeadas reconhecidas incorretamente

| | Texto |
|---|---|
| **Fala esperada** | `I transferred money to Jon at Citi.` |
| **Possível transcrição ASR** | `i transferred money to john at city` |

**Problemas para PLN:**
- `Jon` e `John` podem se referir a pessoas distintas.
- `Citi` (organização) vira `city` (substantivo comum).
- NER e sistemas transacionais podem mapear a entidade errada.

---

**Observações gerais:**
- Áudio com ruído, baixa qualidade ou sobreposição de fala piora a transcrição.
- Em produção, convém adicionar pós-processamento: pontuação, normalização e validação de entidades.